# 🚀 Notebook 6: End-to-End Pipeline Demo

## Complete Pipeline: From Raw MRI to Benign/Malignant Prediction

---

This notebook demonstrates the **complete pipeline** in a single, self-contained script. It combines all steps from Notebooks 01-05 for quick demonstration and inference.

### Complete Workflow
```
┌──────────────┐    ┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│  MRI Brain  │─▶│  Grayscale   │─▶│ Resize 10×10 │─▶│  Row-mean   │
│  Image      │    │  Conversion  │    │              │    │  → 1D       │
└──────────────┘    └──────────────┘    └──────────────┘    └──────────────┘
                                                              │
                                                              ▼
┌──────────────┐    ┌──────────────┐    ┌──────────────┐
│  Prediction │◀─│  CNN Model   │◀─│ DE/IDE with  │
│  Benign /   │    │              │    │ Euler & RK4  │
│  Malignant  │    │              │    │              │
└──────────────┘    └──────────────┘    └──────────────┘
```

## 1. Imports & Configuration

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import resize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, roc_curve, auc, confusion_matrix
from tqdm import tqdm
import warnings

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

# Configuration
TARGET_SIZE = (10, 10)
N_SUBJECTS = 80           # Number of synthetic subjects
SLICES_PER_SUBJECT = 15   # Slices per subject
EPOCHS = 60
BATCH_SIZE = 32

OUTPUT_DIR = './end_to_end_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Environment ready")
print(f"   TensorFlow: {tf.__version__}")
print(f"   GPU: {len(tf.config.list_physical_devices('GPU'))} available")

## 2. Data Generation (Synthetic BraTS-like Data)

Generate synthetic multi-modal MRI brain slices for demonstration.  
**Replace this with real BraTS 2020 data loading for actual experiments.**

In [ ]:
def generate_mri_dataset(n_subjects=80, slices_per_subject=15):
    """Generate synthetic MRI brain slices with tumor labels."""
    H, W = 240, 240
    y_grid, x_grid = np.ogrid[-1:1:H*1j, -1:1:W*1j]
    brain_mask = (x_grid**2 / 0.7**2 + y_grid**2 / 0.9**2) < 1
    
    all_slices = []  # (H, W, 4) multi-modal slices
    all_labels = []  # 0=Benign, 1=Malignant
    
    for subj in tqdm(range(n_subjects), desc="Generating MRI data"):
        is_hgg = np.random.random() < 0.7  # 70% HGG
        label = 1 if is_hgg else 0
        
        for s in range(slices_per_subject):
            base = np.random.uniform(0.25, 0.65)
            channels = []
            
            for mod_scale in [1.0, 0.8, 0.9, 1.1]:  # FLAIR, T1, T1ce, T2
                ch = np.zeros((H, W), dtype=np.float32)
                ch[brain_mask] = base * mod_scale + np.random.normal(0, 0.04, brain_mask.sum())
                
                # Add tumor
                tcx, tcy = 120 + np.random.randint(-25, 25), 120 + np.random.randint(-25, 25)
                tr = np.random.randint(18, 38) if is_hgg else np.random.randint(6, 18)
                tr = max(4, int(tr * (1.1 - abs(s - slices_per_subject/2) / slices_per_subject)))
                
                tumor = ((np.arange(W) - tcx)**2 + (np.arange(H)[:, None] - tcy)**2) < tr**2
                tumor = tumor & brain_mask
                enh = np.random.uniform(1.2, 1.5) if is_hgg else np.random.uniform(1.05, 1.2)
                ch[tumor] *= enh
                channels.append(np.clip(ch, 0, 1))
            
            all_slices.append(np.stack(channels, axis=-1))
            all_labels.append(label)
    
    return all_slices, np.array(all_labels)


mri_slices, labels = generate_mri_dataset(N_SUBJECTS, SLICES_PER_SUBJECT)
print(f"\n📊 Dataset: {len(mri_slices)} slices, Malignant={np.sum(labels==1)}, Benign={np.sum(labels==0)}")

## 3. Preprocessing: Grayscale → Resize → 1D Signal

In [ ]:
def preprocess_to_1d(multi_modal_slice, target_size=(10, 10)):
    """Complete preprocessing: Grayscale → Resize → Row-mean."""
    # Grayscale (mean across modalities)
    gray = np.mean(multi_modal_slice, axis=2)
    if gray.max() > gray.min():
        gray = (gray - gray.min()) / (gray.max() - gray.min())
    
    # Resize to 10x10
    resized = resize(gray, target_size, anti_aliasing=True, preserve_range=True).astype(np.float32)
    
    # Row-wise mean → 1D signal
    signal_1d = np.mean(resized, axis=1)
    return signal_1d


# Process all slices
signals = np.array([preprocess_to_1d(s, TARGET_SIZE) for s in tqdm(mri_slices, desc="Preprocessing")])
print(f"✅ Signals shape: {signals.shape}")

## 4. Differential Equation Modeling

In [ ]:
def numerical_derivative(signal, dt=1.0):
    """Central differences."""
    N = len(signal)
    deriv = np.zeros(N)
    deriv[0] = (signal[1] - signal[0]) / dt
    for i in range(1, N - 1):
        deriv[i] = (signal[i+1] - signal[i-1]) / (2 * dt)
    deriv[-1] = (signal[-1] - signal[-2]) / dt
    return deriv


def fit_ode(signal, dt=1.0):
    """Fit ODE: dy/dt = alpha*y + beta*t + gamma."""
    N = len(signal)
    t = np.arange(N, dtype=np.float64) * dt
    y = signal.astype(np.float64)
    dydt = numerical_derivative(y, dt)
    A = np.column_stack([y, t, np.ones(N)])
    coeffs = np.linalg.lstsq(A, dydt, rcond=None)[0]
    return {'alpha': coeffs[0], 'beta': coeffs[1], 'gamma': coeffs[2]}


def euler_solve(signal, params, dt=1.0):
    """Euler method for ODE."""
    a, b, g = params['alpha'], params['beta'], params['gamma']
    N = len(signal)
    y = np.zeros(N)
    y[0] = signal[0]
    for i in range(N - 1):
        y[i+1] = y[i] + dt * (a * y[i] + b * i * dt + g)
    return y


def rk4_solve(signal, params, dt=1.0):
    """RK4 method for ODE."""
    a, b, g = params['alpha'], params['beta'], params['gamma']
    def f(y, t): return a * y + b * t + g
    N = len(signal)
    y = np.zeros(N)
    y[0] = signal[0]
    for i in range(N - 1):
        t = i * dt
        k1 = dt * f(y[i], t)
        k2 = dt * f(y[i] + k1/2, t + dt/2)
        k3 = dt * f(y[i] + k2/2, t + dt/2)
        k4 = dt * f(y[i] + k3, t + dt)
        y[i+1] = y[i] + (k1 + 2*k2 + 2*k3 + k4) / 6
    return y


def euler_ide_solve(signal, params, lam=0.05, mu=0.3, dt=1.0):
    """Euler method for IDE with memory integral."""
    a, b, g = params['alpha'], params['beta'], params['gamma']
    N = len(signal)
    y = np.zeros(N)
    y[0] = signal[0]
    for i in range(N - 1):
        t_val = i * dt
        ode = a * y[i] + b * t_val + g
        # Integral term
        integral = 0.0
        for j in range(i + 1):
            integral += np.exp(-mu * (i - j) * dt) * y[j] * dt
        y[i+1] = y[i] + dt * (ode + lam * integral)
    return y


def rk4_ide_solve(signal, params, lam=0.05, mu=0.3, dt=1.0):
    """RK4 method for IDE."""
    a, b, g = params['alpha'], params['beta'], params['gamma']
    N = len(signal)
    y = np.zeros(N)
    y[0] = signal[0]
    for i in range(N - 1):
        t_val = i * dt
        integral = sum(np.exp(-mu * (i - j) * dt) * y[j] * dt for j in range(i + 1))
        def f(yv, tv): return a * yv + b * tv + g + lam * integral
        k1 = dt * f(y[i], t_val)
        k2 = dt * f(y[i]+k1/2, t_val+dt/2)
        k3 = dt * f(y[i]+k2/2, t_val+dt/2)
        k4 = dt * f(y[i]+k3, t_val+dt)
        y[i+1] = y[i] + (k1 + 2*k2 + 2*k3 + k4) / 6
    return y


def extract_all_features(signal):
    """Extract complete DE/IDE feature vector."""
    signal = signal.astype(np.float64)
    params = fit_ode(signal)
    
    euler_ode = euler_solve(signal, params)
    rk4_ode = rk4_solve(signal, params)
    euler_ide = euler_ide_solve(signal, params)
    rk4_ide = rk4_ide_solve(signal, params)
    
    features = np.concatenate([
        [params['alpha'], params['beta'], params['gamma']],
        euler_ode, rk4_ode,
        signal - euler_ode, signal - rk4_ode,
        euler_ide, rk4_ide,
        signal - euler_ide, signal - rk4_ide,
        [np.sqrt(np.mean((signal - euler_ode)**2)),
         np.sqrt(np.mean((signal - rk4_ode)**2)),
         np.sqrt(np.mean((signal - euler_ide)**2)),
         np.sqrt(np.mean((signal - rk4_ide)**2)),
         np.sum(signal**2), signal[-1] - signal[0],
         np.mean(np.abs(np.diff(signal, n=2)))]
    ])
    return features


# Extract features for all signals
features_list = []
for sig in tqdm(signals, desc="DE Feature Extraction"):
    try:
        feat = extract_all_features(sig)
        features_list.append(feat)
    except:
        features_list.append(np.zeros(90))

features = np.nan_to_num(np.array(features_list, dtype=np.float32))
print(f"✅ Features shape: {features.shape}")

## 5. Normalize, Split & Reshape

In [ ]:
# Normalize
scaler = StandardScaler()
features_norm = scaler.fit_transform(features)

# Split
X_trainval, X_test, y_trainval, y_test = train_test_split(
    features_norm, labels, test_size=0.15, random_state=42, stratify=labels
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.176, random_state=42, stratify=y_trainval
)

# Reshape for 1D CNN
n_feat = features.shape[1]
TIMESTEPS = 10
pad_size = (TIMESTEPS - n_feat % TIMESTEPS) % TIMESTEPS
CHANNELS = (n_feat + pad_size) // TIMESTEPS

def reshape_cnn(X):
    X_pad = np.pad(X, ((0, 0), (0, pad_size)), mode='constant') if pad_size > 0 else X
    return X_pad.reshape(-1, TIMESTEPS, CHANNELS)

X_train_cnn = reshape_cnn(X_train)
X_val_cnn = reshape_cnn(X_val)
X_test_cnn = reshape_cnn(X_test)

print(f"Train: {X_train_cnn.shape}, Val: {X_val_cnn.shape}, Test: {X_test_cnn.shape}")

# Class weights
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {i: w for i, w in enumerate(cw)}

## 6. Build & Train CNN

In [ ]:
# Build Model
model = models.Sequential([
    layers.Input(shape=(TIMESTEPS, CHANNELS)),
    layers.Conv1D(64, 3, padding='same'), layers.BatchNormalization(), layers.Activation('relu'),
    layers.Conv1D(128, 3, padding='same'), layers.BatchNormalization(), layers.Activation('relu'),
    layers.Dropout(0.3),
    layers.Conv1D(256, 3, padding='same'), layers.BatchNormalization(), layers.Activation('relu'),
    layers.Dropout(0.3),
    layers.GlobalAveragePooling1D(),
    layers.Dense(128, activation='relu'), layers.Dropout(0.5),
    layers.Dense(64, activation='relu'), layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary()

# Train
history = model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_auc', patience=12, restore_best_weights=True, mode='max'),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6)
    ],
    verbose=1
)

print(f"\n✅ Best Val AUC: {max(history.history['val_auc']):.4f}")

## 7. Evaluate & Visualize Results

In [ ]:
# Predict
y_pred_proba = model.predict(X_test_cnn, verbose=0).flatten()
y_pred = (y_pred_proba >= 0.5).astype(int)

# Metrics
print("\n" + "="*50)
print(" TEST SET RESULTS")
print("="*50)
print(classification_report(y_test, y_pred, target_names=['Benign', 'Malignant']))

# Plots
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Training curves
epochs = range(1, len(history.history['loss']) + 1)
axes[0, 0].plot(epochs, history.history['loss'], 'b-', lw=2, label='Train')
axes[0, 0].plot(epochs, history.history['val_loss'], 'r-', lw=2, label='Val')
axes[0, 0].set_title('Loss Curves', fontsize=13, fontweight='bold')
axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

# 2. AUC curves
axes[0, 1].plot(epochs, history.history['auc'], 'b-', lw=2, label='Train')
axes[0, 1].plot(epochs, history.history['val_auc'], 'r-', lw=2, label='Val')
axes[0, 1].set_title('AUC Curves', fontsize=13, fontweight='bold')
axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

# 3. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
im = axes[0, 2].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        axes[0, 2].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=16, fontweight='bold')
axes[0, 2].set_xticks([0, 1]); axes[0, 2].set_xticklabels(['Benign', 'Malignant'])
axes[0, 2].set_yticks([0, 1]); axes[0, 2].set_yticklabels(['Benign', 'Malignant'])
axes[0, 2].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0, 2].set_xlabel('Predicted'); axes[0, 2].set_ylabel('True')

# 4. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)
axes[1, 0].plot(fpr, tpr, 'darkorange', lw=2.5, label=f'AUC = {roc_auc:.3f}')
axes[1, 0].plot([0, 1], [0, 1], 'k--'); axes[1, 0].fill_between(fpr, tpr, alpha=0.1, color='darkorange')
axes[1, 0].set_title('ROC Curve', fontsize=13, fontweight='bold')
axes[1, 0].legend(fontsize=11); axes[1, 0].grid(True, alpha=0.3)

# 5. Prediction Distribution
axes[1, 1].hist(y_pred_proba[y_test==0], bins=20, alpha=0.6, color='seagreen', label='Benign', density=True)
axes[1, 1].hist(y_pred_proba[y_test==1], bins=20, alpha=0.6, color='crimson', label='Malignant', density=True)
axes[1, 1].axvline(0.5, color='black', linestyle='--', lw=2)
axes[1, 1].set_title('Prediction Distribution', fontsize=13, fontweight='bold')
axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

# 6. Sample signals with predictions
t = np.arange(10)
for i in range(min(5, len(X_test))):
    color = 'crimson' if y_pred[i] == 1 else 'seagreen'
    true_label = 'M' if y_test[i] == 1 else 'B'
    pred_label = 'M' if y_pred[i] == 1 else 'B'
    correct = '✅' if y_test[i] == y_pred[i] else '❌'
    axes[1, 2].plot(t, signals[i][:10], 'o-', color=color, alpha=0.6, 
                   label=f'{correct} True:{true_label} Pred:{pred_label}' if i < 5 else '')
axes[1, 2].set_title('Sample Signals + Predictions', fontsize=13, fontweight='bold')
axes[1, 2].legend(fontsize=8); axes[1, 2].grid(True, alpha=0.3)

fig.suptitle('🧠 Brain Tumor Classification - End-to-End Results', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'end_to_end_results.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n🎯 Final AUC-ROC: {roc_auc:.4f}")
print(f"💾 Results saved to: {OUTPUT_DIR}")

## 8. Single Prediction Demo

Demonstrate how to make a prediction for a **single new MRI slice**.

In [ ]:
def predict_single_slice(multi_modal_slice, model, scaler, timesteps=10, channels=9):
    """
    Complete pipeline for a single slice prediction.
    Input: multi-modal MRI slice (H, W, 4)
    Output: prediction dict with label and probability
    """
    # Step 1: Preprocess to 1D signal
    signal = preprocess_to_1d(multi_modal_slice)
    
    # Step 2: Extract DE/IDE features
    features = extract_all_features(signal)
    features = np.nan_to_num(features.reshape(1, -1), nan=0.0)
    
    # Step 3: Normalize
    features_scaled = scaler.transform(features)
    
    # Step 4: Reshape for CNN
    n_f = features_scaled.shape[1]
    pad = (timesteps - n_f % timesteps) % timesteps
    features_pad = np.pad(features_scaled, ((0, 0), (0, pad)), mode='constant')
    n_ch = features_pad.shape[1] // timesteps
    features_cnn = features_pad.reshape(1, timesteps, n_ch)
    
    # Step 5: Predict
    probability = model.predict(features_cnn, verbose=0).flatten()[0]
    label = 'Malignant (HGG)' if probability >= 0.5 else 'Benign (LGG)'
    confidence = probability if probability >= 0.5 else 1 - probability
    
    return {
        'label': label,
        'probability': float(probability),
        'confidence': float(confidence),
        'signal': signal
    }


# Demo with a random slice
demo_slice = mri_slices[42]  # Pick a sample
result = predict_single_slice(demo_slice, model, scaler)

print(f"🧠 PREDICTION RESULT:")
print(f"   Label      : {result['label']}")
print(f"   Probability: {result['probability']:.4f}")
print(f"   Confidence : {result['confidence']:.1%}")

# Visualize
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))

ax1.imshow(np.mean(demo_slice, axis=2), cmap='gray')
ax1.set_title('Input MRI (Grayscale)', fontsize=12, fontweight='bold')
ax1.axis('off')

ax2.plot(result['signal'], 'o-', color='steelblue', linewidth=2, markersize=8)
ax2.set_title('1D Signal (Row-mean)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Row Index'); ax2.grid(True, alpha=0.3)

color = 'crimson' if result['probability'] >= 0.5 else 'seagreen'
ax3.barh(['Prediction'], [result['probability']], color=color, height=0.4, alpha=0.8)
ax3.barh(['Prediction'], [1-result['probability']], left=[result['probability']], 
         color='lightgray', height=0.4, alpha=0.5)
ax3.axvline(0.5, color='black', linestyle='--', linewidth=2)
ax3.set_xlim(0, 1)
ax3.text(0.5, -0.15, f"\u2190 Benign | Malignant \u2192", ha='center', transform=ax3.transAxes, fontsize=10)
ax3.set_title(f'{result["label"]} ({result["confidence"]:.1%})', fontsize=13, fontweight='bold', color=color)

plt.tight_layout()
plt.show()

---

## ✅ End-to-End Pipeline Complete!

This notebook demonstrated the **entire** brain tumor classification pipeline:

1. ✅ **Data Loading** - Synthetic BraTS-like MRI data generation
2. ✅ **Preprocessing** - Grayscale → Resize 10×10 → Row-mean → 1D signal
3. ✅ **DE/IDE Modeling** - ODE fitting + Euler/RK4 solving + Feature extraction
4. ✅ **CNN Training** - 1D CNN with BatchNorm, Dropout, class weighting
5. ✅ **Evaluation** - ROC, confusion matrix, prediction distribution
6. ✅ **Inference** - Single-slice prediction demo

### To use with real BraTS 2020 data:
1. Download from [Kaggle](https://www.kaggle.com/datasets/awsaf49/brats2020-training-data)
2. Replace the `generate_mri_dataset()` function with NIfTI/H5 loading
3. Use `name_mapping.csv` for HGG/LGG grade labels
4. Run the individual notebooks (01-05) for detailed analysis